# Finding α

In [88]:
from scipy.optimize import minimize
import numpy as np

def s_curve(delta, p,  delta_0):
    return -((delta - p) / (1 + np.exp(-0.7 + 3.1 / (delta_0) * delta)))



In [89]:
def estimating_parameters(h = 0.01 , delta_0 = 0.09):
  p = 0
  initial_guess = 0.09
  result = minimize(lambda x: s_curve(x, p , delta_0), initial_guess)
  f_0  = -result.fun

  p = h
  result = minimize(lambda x: s_curve(x, p , delta_0), initial_guess)
  f_h_plus  = -result.fun

  p = -h
  result = minimize(lambda x: s_curve(x, p , delta_0), initial_guess)
  f_h_minus  = -result.fun

  first_derviate = (f_h_plus - f_h_minus)/(2*h)

  second_derviate = (f_h_plus + f_h_minus - 2*f_0)/(h**2)

  return f_0 , first_derviate , second_derviate





In [90]:
α =  torch.tensor( estimating_parameters(h = 0.01))

In [91]:
α

tensor([ 0.0135, -0.3176,  5.0787])

# Dynamic

In [92]:
import numpy as np

def two_combinations(vector):
    """
    Convert an n-value vector to an n^2 2-dimensional vector representing all 2-combinations.

    :param vector: An n-value vector (list or numpy array).
    :return: An n^2 2-dimensional vector (2D numpy array).
    """
    n = len(vector)
    combination_vectors = []

    for i in range(n):
        for j in range(n):
            combination_vectors.append([vector[i], vector[j]])

    return np.array(combination_vectors)


import numpy as np
from scipy.stats import expon

def simulate_markov_batch(Q, initial_states, times, num_simulations):
    num_states = len(Q)
    num_times = len(times)
    states_at_times = np.zeros((num_simulations, num_times), dtype=int)

    for sim in range(num_simulations):
        state = initial_states
        states_at_times[sim, 0] = state

        for i in range(1, num_times):
            current_time = times[i - 1]
            end_time = times[i]
            while current_time < end_time:
                rate = -Q[state, state]
                time_to_next = expon.rvs(scale = 1/rate)
                #print(time_to_next)


                current_time += time_to_next
                if current_time < end_time:
                    transition_probs = Q[state, :] / rate
                    transition_probs[state] = 0
                    state = np.random.choice(num_states, p=transition_probs)

            states_at_times[sim, i] = state

    return states_at_times



def simulate_markov_batch_1(Q, initial_states, times, num_simulations , delta_t):
    num_states = len(Q)
    num_times = len(times)
    states_at_times = np.zeros((num_simulations, num_times), dtype=int)

    for sim in range(num_simulations):
        state = initial_states
        states_at_times[sim, 0] = state
        q = Q * delta_t
        I = np.identity(num_states)
        p = np.zeros((1,num_states))
        p[0,state] = 1
        for i in range(1, num_times):
          p = np.matmul(p,q+I)
          #print(p)
          state = np.random.choice(num_states, p=np.squeeze(p))
          states_at_times[sim, i] = state

    return states_at_times

import torch
import torch.nn.functional as F
from torch import nn, Tensor
import numpy as np
from torch.nn import Module, Linear, BatchNorm1d, Tanh
from numba import cuda
from torch import autograd
import torch.nn as nn
import torch.nn.functional as F
import math
"""
In this section we have used code available online in : https://github.com/frankhan91/DeepBSDE

"""

class StochasticProcess:
    """
    Base class for defining PDE related function.

    Args:
    eqn_config (dict): dictionary containing PDE configuration parameters

    Attributes:
    dim (int): dimensionality of the problem
    total_time (float): total time horizon
    num_time_interval (int): number of time steps
    delta_t (float): time step size
    sqrt_delta_t (float): square root of time step size
    y_init (None): initial value of the function
    """

    def __init__(self, eqn_config: dict):
        self.dim = eqn_config['dim']
        self.total_time = eqn_config['total_time']
        self.num_time_interval = eqn_config['num_time_interval']
        self.delta_t = self.total_time / self.num_time_interval
        self.sqrt_delta_t = np.sqrt(self.delta_t)
        self.y_init = None

    def sample(self, num_sample: int) -> Tensor:
        """
        Sample forward SDE.

        Args:
        num_sample (int): number of samples to generate

        Returns:
        Tensor: tensor of size [num_sample, dim+1] containing samples
        """
        raise NotImplementedError

    def r_u(self, t: float, x: Tensor, y: Tensor, z: Tensor) -> Tensor:
        """
        Interest rate in the PDE.

        Args:
        t (float): current time
        x (Tensor): tensor of size [batch_size, dim] containing space coordinates
        y (Tensor): tensor of size [batch_size, 1] containing function values
        z (Tensor): tensor of size [batch_size, dim] containing gradients

        Returns:
        Tensor: tensor of size [batch_size, 1] containing generator values
        """
        raise NotImplementedError

    def h_z(self, t,x,y,z: Tensor) -> Tensor:
        """
        Function to compute H(z) in the PDE.

        Args:
        h (float): value of H function
        z (Tensor): tensor of size [batch_size, dim] containing gradients

        Returns:
        Tensor: tensor of size [batch_size, dim] containing H(z)
        """
        raise NotImplementedError

    def terminal(self, x: Tensor) -> Tensor:
        """
        Terminal condition of the PDE.

        Args:
        t (float): current time
        x (Tensor): tensor of size [batch_size, dim] containing space coordinates

        Returns:
        Tensor: tensor of size [batch_size, 1] containing terminal values
        """
        raise NotImplementedError


class RFQ(StochasticProcess):
  """
  Args:
  eqn_config (dict): dictionary containing PDE configuration parameters
  """
  def __init__(self, basic_config,specific_config):
    super(RFQ, self).__init__(basic_config)

    self.n_liqiudity_state  = specific_config['nls']
    self.x_init = np.ones(self.dim) * specific_config['init']
    self.lamdas = specific_config['lamdas']
    self.lamda_initial_state = specific_config['init_state'] #integer
    self.sigma = specific_config['sigma']
    self.k = specific_config['k']
    self.Q = specific_config['Q']

  def sample(self, num_sample)-> tuple:
    """
    Sample forward SDE.

    Args:
    num_sample (int): number of samples to generate

    Returns:
    tuple: tuple of two tensors: dw_sample of size [num_sample, dim, num_time_interval] and
    x_sample of size [num_sample, dim, num_time_interval+1]
    """

    dw_sample = np.random.normal(size=[num_sample,  self.num_time_interval]) * self.sqrt_delta_t
    x_sample = np.zeros([num_sample, self.num_time_interval + 1])
    x_sample[:, 0] = np.ones(num_sample) * self.x_init

    select_Q = np.ones([num_sample, self.n_liqiudity_state**2  , self.n_liqiudity_state**2  ]) * np.expand_dims(np.exp(self.Q * self.delta_t), axis=0)
    select_lamda = np.ones([num_sample, self.n_liqiudity_state**2 , 2]) * np.expand_dims(two_combinations(self.lamdas), axis=0)
    #lamda_process = simulate_markov_batch(self.Q ,self.lamda_initial_state,np.array([i*self.delta_t for i in range(self.num_time_interval)]) ,num_sample )
    lamda_process = simulate_markov_batch_1(self.Q ,self.lamda_initial_state,np.array([i*self.delta_t for i in range(self.num_time_interval)]) ,num_sample ,  self.delta_t)

    ask_lamda = np.array([[self.lamdas[x // 2]for x in y] for y in  lamda_process ])
    bid_lamda = np.array([[self.lamdas[x % 2]for x in y] for y in  lamda_process ])

    for i in range(self.num_time_interval):

        x_sample[:,  i + 1] =  x_sample[:,  i ]+ (ask_lamda[:, i] -  bid_lamda[:, i]) * self.k * self.delta_t + self.sigma * dw_sample[:, i]
    return ask_lamda , bid_lamda, x_sample , lamda_process

  def r_u(self, t, x, y, z)-> torch.Tensor:
    """
    Interest rate in the PDE.

    Args:
    t (float): current time
    x (torch.Tensor): tensor of size [batch_size, dim] containing space coordinates
    y (torch.Tensor): tensor of size [batch_size, 1] containing function values
    z (torch.Tensor): tensor of size [batch_size, dim] containing gradients

    Returns:
    torch.Tensor: tensor of size [batch_size, 1] containing generator values
    """

    return 0

  def h_z(self,t,x,y,z)-> torch.Tensor:
      """
      Function to compute $h^T Z$ in the PDE.

      Args:
      t (float): current time
      x (torch.Tensor): tensor of size [batch_size, dim] containing space coordinates
      y (torch.Tensor): tensor of size [batch_size, 1] containing function value
      z (torch.Tensor): tensor of size [batch_size, dim] containing gradients

      Returns:
      torch.Tensor: tensor of size [batch_size, 1] containing H(z)
      """
      return 0


  def terminal(self,  x)-> torch.Tensor:
    """
    Terminal condition of the PDE.

    Args:
    t (float): current time
    x (torch.Tensor): tensor of size [batch_size, dim] containing space coordinates

    Returns:
    torch.Tensor: tensor of size [batch_size, 1] containing terminal values
    """
    return 0

"""A trading environment"""
import gym
from gym import spaces
from gym.utils import seeding

import numpy as np




class BoudEnv(gym.Env):
    """
    trading environment;
    """

    # trade_freq in unit of day, e.g 2: every 2 day; 0.5 twice a day;
    def __init__(self,basic_config,specific_config, num_sim=100):

        # simulated data: array of asset price, option price and delta paths (num_path x num_period)
        # generate data now
        self.sp = RFQ(basic_config,specific_config)



    def sample(self, batch_size):
      return  self.sp.sample(batch_size)




# DGM

In [127]:
import torch
import torch.nn as nn


def loss(
    θ: nn.Module,
    q: torch.tensor,
    t: torch.tensor,
    T: float,
    Q: torch.tensor,
    κ: float,
    γ: float,
    σ: float,
    z: float,
    j: torch.tensor,
    α_a: torch.tensor,
    α_b: torch.tensor,
    λ_a: torch.tensor,
    λ_b: torch.tensor,
    batch_size: int
) -> float:
    t1  = torch.tensor(t, requires_grad=True)
    dt = torch.autograd.grad(torch.sum(θ(t1, q,j)), t1)[0]

    Q_sum = 0
    for k_a in range(4):

      Q_sum += Q[torch.arange(batch_size), torch.squeeze(j),k_a * torch.ones(batch_size).to(torch.int) ] * torch.squeeze(θ(t, q,k_a * torch.ones(t.size()[0] , 1)))

    print(Q_sum.size())

    return (
        torch.squeeze(dt)
        + torch.squeeze(κ * (λ_a - λ_b) * q)
        - torch.squeeze(1 / 2 * γ * σ**2 * q**2)
        + Q_sum
        + z * torch.squeeze(λ_b * α_b[0] + λ_a * α_a[0])
        + torch.squeeze(
            λ_b * α_b[1] * (θ(t, q ,j) - θ(t, q + z,j))
            + λ_a * α_a[1] * (θ(t, q,j) - θ(t, q - z,j))
        )
        + (1 / (2 * z))
        * torch.squeeze(
            λ_b * α_b[2] * (θ(t, q,j) - θ(t, q + z,j)) ** 2
            + λ_a * α_a[2] * (θ(t, q,j) - θ(t, q - z,j)) ** 2
        )
    )


class NeuralBox(nn.Module):
    def __init__(self, indim=100, outdim=50):
        super().__init__()
        self.activation = nn.Tanh()
        self.z = nn.Linear(outdim, outdim, bias=False)
        self.g = nn.Linear(outdim, outdim, bias=False)
        self.r = nn.Linear(outdim, outdim, bias=False)
        self.h = nn.Linear(outdim, outdim, bias=False)
        self.z1 = nn.Linear(outdim, outdim)
        self.g1 = nn.Linear(outdim, outdim)
        self.r1 = nn.Linear(outdim, outdim)
        self.h1 = nn.Linear(outdim, outdim)
        self.s1 = nn.Linear(indim, outdim)
        self.s2 = nn.Linear(indim, outdim)
        self.s3 = nn.Linear(indim, outdim)
        self.s4 = nn.Linear(indim, outdim)

    def forward(self, x, s):
        z1 = self.z(s) + self.s1(x)
        z1 = self.activation(z1)
        g1 = self.g(s) + self.s2(x)
        g1 = self.activation(g1)
        r1 = self.r(s) + self.s3(x)
        r1 = self.activation(r1)
        h1 = self.h(torch.mul(s, r1)) + self.s4(x)
        h1 = self.activation(h1)
        s2 = torch.mul((1.0 - g1), h1) + torch.mul(z1, s)
        return s2


class DGM(nn.Module):
    def __init__(
        self,
        dim=1,
        layersize=10,
    ):
        """
        We are minimizing the following function (page 28 of the paper):

        0 = ∂_t θ(t, q) + κ(λ_a - λ_b) q - 1 / 2 * γ σ^2 q^2
            + ∑ Q + θ + z(λ α_b0 + λ α_a0)
            ...

        Also, θ(T, q) should be 0 for all q.

        Ahmad wants a standalone loss function that he can use to create the
        NN.

        - m_a and m_b = 4 (the dimensions of Q)
        - ∂θ - compute using PyTorch
        - ...
        """
        super().__init__()
        self.dim = dim
        self.layer1 = nn.Linear(self.dim, layersize)
        self.module1 = NeuralBox(indim=self.dim, outdim=layersize)
        self.module2 = NeuralBox(indim=self.dim, outdim=layersize)
        self.module3 = NeuralBox(indim=self.dim, outdim=layersize)

        self.last_layer = nn.Linear(layersize, 1)

        self.activation = nn.Tanh()

    def forward(self, y):
        # model inputs are t and Q
        s1 = self.layer1(y)
        s1 = self.activation(s1)
        s2 = self.module1(y, s1)
        s3 = self.module2(y, s2)
        s4 = self.module3(y, s3)
        out = self.last_layer(s4)
        return out


class Theta(nn.Module):
    def __init__(
        self,
        dim=3,
        layersize=10,
    ):
        """
        We are minimizing the following function (page 28 of the paper):

        0 = ∂_t θ(t, q) + κ(λ_a - λ_b) q - 1 / 2 * γ σ^2 q^2
            + ∑ Q + θ + z(λ α_b0 + λ α_a0)
            ...

        Also, θ(T, q) should be 0 for all q.

        Ahmad wants a standalone loss function that he can use to create the
        NN.

        - m_a and m_b = 4 (the dimensions of Q)
        - ∂θ - compute using PyTorch
        - ...
        """
        super().__init__()
        self.dim = dim
        self.model = DGM(dim = 3)




    def forward(self, t,q,j):
        # model inputs are t and Q
        out = self.model(torch.cat((t,q,j) , dim = 1))
        return out

In [109]:
Q = np.array([[-14.01, 4.37,4.37,5.27] , [19.32, -60.91,12.54,29.05] , [19.32,12.54,-60.91,29.05] , [23.67,15.00,15.00,-53.67]])

basic = {'dim' : 1 , 'total_time' : 0.25 , 'num_time_interval': 90 }
specific = {'init':103.593 , 'sigma':0.1839 ,  'nls' :2 , 'lamdas' : np.array([10.83, 73.03]) /10, 'init_state':1,'k' : 2.29 , 'Q': Q}

env = BoudEnv(basic , specific)
z = 1
batch_size = 20

In [110]:
Q = (torch.unsqueeze(torch.tensor(Q), dim = 0) * torch.ones(batch_size, 1, 1))

In [111]:
ask_lamda, bid_lamda,prices,states= env.sample(batch_size)
ask_lamda = torch.tensor(ask_lamda).to(torch.float32)
bid_lamda = torch.tensor(bid_lamda).to(torch.float32)
prices = torch.tensor(prices).to(torch.float32)
q = torch.zeros((batch_size ,env.sp.num_time_interval + 1))
ask_delta = torch.zeros((batch_size ,env.sp.num_time_interval))
bid_delta = torch.zeros((batch_size ,env.sp.num_time_interval))
model = Theta()

In [137]:
states[:,90]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


IndexError: index 90 is out of bounds for axis 1 with size 90

In [130]:
env.sp.num_time_interval

90

In [132]:
l = 0
for t1 in range(env.sp.num_time_interval):

    #q[:, t+1] = q[:, t] + z*(bid_lamda[:,t] - ask_lamda[:,t])

    #(z*ask_delta[:,i] * ask_lamda[:,i]* scurve(ask_delta[:,i],alpha , beta, delta_0) + z*bid_delta[:,i] * bid_lamda[:,i]* scurve(bid_delta[:,i],alpha , beta, delta_0))
    q[:, t1+1] = q[:, t1] + z*(bid_lamda[:,t1]* torch.rand(batch_size) - ask_lamda[:,t1]*torch.rand(batch_size))

    l += loss(
    θ = model,
    q =  q[:, t1:t1+1] ,
    t = t1 * torch.ones(batch_size , 1) * env.sp.delta_t,
    T = 1,
    Q = Q,
    κ = 2.29,
    γ = 0,
    σ = 0.1839,
    z = 1,
    j = torch.tensor(states[:, t1:t1+1] ),
    α_a = α,
    α_b = α,
    λ_a = ask_lamda[:,t1:t1+1],
    λ_b = bid_lamda[:,t1:t1+1],
    batch_size = batch_size
    )
for i in range(4):
  T = env.sp.num_time_interval
  l+= torch.square(model(T * torch.ones(batch_size , 1) * env.sp.delta_t ,q[:, T:T+1]  , i * torch.ones(batch_size , 1)))






/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-127-03edf6c6f11f>:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t1  = torch.tensor(t, requires_grad=True)


torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([20

In [133]:
env.sp.delta_t * 90

0.25

In [73]:
import torch

# Example 3D tensor (batch of 2D matrices)
batch_size = 4
num_rows = 3
num_columns = 3
A = torch.rand(batch_size, num_rows, num_columns)

# Example row and column indices (1D tensors)
rows = torch.tensor([0, 1, 2, 1])  # Indices for rows to be selected from each batch
cols = torch.tensor([2, 1, 0, 2])  # Indices for columns to be selected from each batch

# Selecting the specified elements
selected_elements = A[torch.arange(batch_size), rows, cols]

print("Selected Elements:", selected_elements)


Selected Elements: tensor([0.6511, 0.2053, 0.6887, 0.1890])
